In [9]:
# Import libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import plotly as py
from tqdm import tqdm  # Import tqdm for the progress bar
from plotly import graph_objs as go

# ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [10]:
## Define file paths for the different phases
# Get the current working directory
dir_path = "/Users/anupmeshram/code/Enterprise Forecasting/"

# Append the parent directory to sys.path
sys.path.append(os.path.join(dir_path, "./"))

# File path to the data file
phase_0_price_file = os.path.join(dir_path, 'data', "Phase 0 - Price.csv")  
phase_0_sales_file = os.path.join(dir_path, 'data', "Phase 0 - Sales.csv")
phase_1_price_file = os.path.join(dir_path, 'data', "Phase 1 - Price.csv")
phase_1_sales_file = os.path.join(dir_path, 'data', "Phase 1 - Sales.csv")
print(phase_0_price_file)
print(phase_0_sales_file)
print(phase_1_price_file)
print(phase_1_sales_file)


/Users/anupmeshram/code/Enterprise Forecasting/data/Phase 0 - Price.csv
/Users/anupmeshram/code/Enterprise Forecasting/data/Phase 0 - Sales.csv
/Users/anupmeshram/code/Enterprise Forecasting/data/Phase 1 - Price.csv
/Users/anupmeshram/code/Enterprise Forecasting/data/Phase 1 - Sales.csv


In [11]:
# ---------- PHASE 0 DATA PROCESSING ----------
# Read Phase 0 price data
price_phase_0 = pd.read_csv(phase_0_price_file, na_values=np.nan)
price_phase_0["Value"] = "Price"
price_phase_0 = price_phase_0.set_index(["Client", "Warehouse", "Product", "Value"]).stack()

# Read Phase 0 sales data
sales_phase_0 = pd.read_csv(phase_0_sales_file, na_values=np.nan)
sales_phase_0["Value"] = "Sales"
sales_phase_0 = sales_phase_0.set_index(["Client", "Warehouse", "Product", "Value"]).stack()

# Combine Phase 0 price and sales data
df_phase_0 = pd.concat([price_phase_0, sales_phase_0]).unstack("Value").reset_index()
df_phase_0.columns = ["Client", "Warehouse", "Product", "ds", "Price", "y"]
df_phase_0["ds"] = pd.to_datetime(df_phase_0["ds"])
df_phase_0 = df_phase_0.astype({"Price": np.float32, "y": np.float32, "Client": "category", "Warehouse": "category", "Product": "category"})
df_phase_0['y'] = pd.to_numeric(df_phase_0['y'], errors='coerce')
df_phase_0 = df_phase_0.sort_values(["Client", "Warehouse", "Product", "ds"])
df_phase_0["client_warehouse_product_id"] = df_phase_0["Client"].astype(str) + "_" + df_phase_0["Warehouse"].astype(str) + "_" + df_phase_0["Product"].astype(str)
df_phase_0["Phase"] = "Phase 0"

# ---------- PHASE 1 DATA PROCESSING ----------

# Read Phase 1 price data
price_phase_1 = pd.read_csv(phase_1_price_file, na_values=np.nan)
price_phase_1["Value"] = "Price"
price_phase_1 = price_phase_1.set_index(["Client", "Warehouse", "Product", "Value"]).stack()

# Read Phase 1 sales data
sales_phase_1 = pd.read_csv(phase_1_sales_file, na_values=np.nan)
sales_phase_1["Value"] = "Sales"
sales_phase_1 = sales_phase_1.set_index(["Client", "Warehouse", "Product", "Value"]).stack()

# Combine Phase 1 price and sales data
df_phase_1 = pd.concat([price_phase_1, sales_phase_1]).unstack("Value").reset_index()
df_phase_1.columns = ["Client", "Warehouse", "Product", "ds", "Price", "y"]
df_phase_1["ds"] = pd.to_datetime(df_phase_1["ds"])
df_phase_1 = df_phase_1.astype({"Price": np.float32, "y": np.float32, "Client": "category", "Warehouse": "category", "Product": "category"})
df_phase_1['y'] = pd.to_numeric(df_phase_1['y'], errors='coerce')
df_phase_1 = df_phase_1.sort_values(["Client", "Warehouse", "Product", "ds"])
df_phase_1["client_warehouse_product_id"] = df_phase_1["Client"].astype(str) + "_" + df_phase_1["Warehouse"].astype(str) + "_" + df_phase_1["Product"].astype(str)
df_phase_1["Phase"] = "Phase 1"

# ---------- COMBINE BOTH PHASES ----------

# Combine Phase 0 and Phase 1 dataframes
multi_phase_sales_data = pd.concat([df_phase_0, df_phase_1], ignore_index=True)

# Sort the dataframe by multiple columns
sorted_df = multi_phase_sales_data.sort_values(
    by=['Client', 'Warehouse', 'Product', 'ds'], 
    ascending=[True, True, True, True]  # Ascending for Client, Warehouse, Product, ds, Descending for Price
)


In [12]:
sorted_df.tail()

,Client,Warehouse,Product,ds,Price,y,client_warehouse_product_id,Phase
2754694,46,318,14294,2023-12-04,NaN,0.0,46_318_14294,Phase 1
2754695,46,318,14294,2023-12-11,46.990002,1.0,46_318_14294,Phase 1
2754696,46,318,14294,2023-12-18,46.990002,1.0,46_318_14294,Phase 1
2754697,46,318,14294,2023-12-25,39.189999,1.0,46_318_14294,Phase 1
2754698,46,318,14294,2024-01-01,45.423336,3.0,46_318_14294,Phase 1


In [13]:
#  Drop the Phase column
sorted_df.drop(columns=['Phase'], inplace=True)

# Export the raw data to parquet file  
sorted_df.to_parquet(os.path.join(dir_path, 'data', "raw_data.parquet"))

In [14]:
#############################
#### Exploration #####
#############################
# Check how many rows have missing prices
missing_prices = sorted_df[sorted_df['Price'].isnull()]
# 1. Overview of Missing Prices
print(f"Total missing Price values: {missing_prices.shape[0]}")

# 2. Group by Client, Warehouse, and Product to see where missing Prices are most common
missing_price_stats = missing_prices.groupby(['Client', 'Warehouse', 'Product']).size().reset_index(name='MissingCount')
missing_price_stats = missing_price_stats.sort_values(by='MissingCount', ascending=False)

print("Top combinations of Client, Warehouse, and Product with the most missing Prices:")
print(missing_price_stats.head(10))

# 3. Percentage of missing Prices for each combination
total_count = sorted_df.groupby(['Client', 'Warehouse', 'Product']).size().reset_index(name='TotalCount')
missing_percentage_stats = pd.merge(missing_price_stats, total_count, on=['Client', 'Warehouse', 'Product'])
missing_percentage_stats['MissingPercentage'] = (missing_percentage_stats['MissingCount'] / missing_percentage_stats['TotalCount']) * 100

print("Combinations of Client, Warehouse, and Product with the highest missing Price percentages:")
print(missing_percentage_stats[['Client', 'Warehouse', 'Product', 'MissingPercentage']].sort_values(by='MissingPercentage', ascending=False).head(10))

# 4. Summary statistics for Price, excluding the missing values
price_summary_stats = sorted_df['Price'].describe()
print("Summary statistics for Prices (excluding missing values):")
print(price_summary_stats)


Total missing Price values: 1946496
Top combinations of Client, Warehouse, and Product with the most missing Prices:
          Client Warehouse Product  MissingCount
33871266       9        82    1024           182
15734034       4        98    6828           182
58514929      16       329    1591           182
72024573      20       221    6671           182
28948705       7       303    6030           182
147325399     41        70    2915           182
10057828       2       251    5109           182
4526539        1        79    2955           182
60216053      17       146    5667           182
64212945      18       177    2638           182
Combinations of Client, Warehouse, and Product with the highest missing Price percentages:
    Client Warehouse Product  MissingPercentage
0        9        82    1024          99.453552
605     20       221   12611          99.453552
607     28       280    5486          99.453552
608     16       305    6722          99.453552
609     17   